In [ ]:
import json
from pathlib import Path

import torch
import h5py
import numpy as np
from ipywidgets import interact
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
# Batch size and dataloaders
from torch.utils.data import Dataset, DataLoader
# TT split
from sklearn.model_selection import train_test_split
# Training

In [ ]:
ID = "cylinder_Re41_raw"
NAME = "2D Cylinder raw"

In [ ]:
DATASET_INPUT = "cylinder_8000"
dataset_path = Path.cwd().parents[1] / f"FluidLearn/data/01_raw/{DATASET_INPUT}"

In [ ]:
### Load data
cases = sorted([file.name for file in dataset_path.iterdir() if file.suffix == ".h5" and file.stem != "constants"], key=lambda x: float(x.split(".")[0]))
N_cases = len(cases)
# chosen_cases = np.linspace(0, N_cases-1, 20).astype(int) # Use 20 evenly spaced cases
# chosen_cases = np.repeat(N_cases-1, 20).astype(int)
chosen_cases = np.repeat(0, 1).astype(int)
# chosen_cases = np.arange(0, N_cases, 1).astype(int) # Use all cases
data_bycase = [] # Structure: case_name -> [scalars, fields], one entry per snapshot
keys = ["p", "Ux", "Uy"]

for case_name in chosen_cases:#range(len(cases))
    case_path = dataset_path / cases[case_name]
    
    with h5py.File(case_path, "r") as file:
        data = {field: file[field][:].astype(np.float32) for field in keys}
        fields = np.stack([data[key] for key in keys], axis=1)  # Shape: (snapshots, fields, cells)
        t = file['t'][:].astype(np.float32)
        
        # Access attributes
        nu = file.attrs['nu'].astype(np.float32)
        Re = 2.0 / nu

    fields[np.isnan(fields)] = 0.0

    t = t[1:]
    fields = fields[1:]

    Nnaps, Nfields = fields.shape[:2]

    dt = t[1:] - t[:-1] # Time to the next snapshot
    dt = np.append(dt, dt[-1]) # The last snapshot has no next one, reuse the previous step

    data_bycase.append([
        np.concatenate([np.tile([Re], (Nnaps, 1)), dt.reshape(-1, 1)], axis=1),  # Scalars
        fields,    # Fields
    ])

# # Separate train from test cases
# test_cases = [len(data_bycase) // 2]
# train_cases = [i for i in range(len(data_bycase)) if i not in test_cases]
# data_bycase_train = [data_bycase[i] for i in train_cases]

data_bycase_train = data_bycase
data_bycase_test = data_bycase

In [ ]:
train = []
for case in data_bycase_train:
    train.append({
        "scalars": case[0],
        "fields": case[1]
    })
train_Re = [case[0][0, 0] for case in data_bycase_train]
N_train = sum([len(case["scalars"]) for case in train])

test = []
for case in data_bycase_test:
    test.append({
        "scalars": case[0],
        "fields": case[1]
    })
test_Re = [case[0][0, 0] for case in data_bycase_test]
N_test = sum([len(case["scalars"]) for case in test])

N_total = N_train + N_test

In [ ]:
output_dir = Path.cwd() / ID
output_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "id": ID,
    "source": f"https://huggingface.co/datasets/spxd/{ID}/tree/main",
    "loader": "huggingface",
    "name": NAME,
    "dataOrigin": {
        "pretrain": "none",
        "train": "synthetic",
        "test": "synthetic"
    },
    "mesh": "Unstructured",
    "problem": "Rollout",
    "compressibility": "Incompressible",
    "boundary": "Fixed",
    "image": "assets/cylinder-flow-dataset.webp",
    "geometry": "assets/meshes/cylinder8000.json",
    "description": "A canonical wake prediction task on the native finite-volume mesh: use a sequence of velocity and pressure snapshots on its cells to predict the future incompressible flow around a cylinder.",
    "reynoldsNumbers": sorted({int(round(float(Re))) for Re in train_Re + test_Re}),
    "inputScalarNames": ["Re", "dt"],
    "flowVariables": keys,
    "splitFractions": {
        "train": round(N_train / N_total, 3),
        "test": round(N_test / N_total, 3)
    },
    "sample_count": int(N_total)
}

print(f"Saving datasets and metadata to {output_dir.resolve()}")
for name, split in [("train", train), ("test", test)]:
    split_dir = output_dir / name
    split_dir.mkdir(parents=True, exist_ok=True)

    Re = test_Re if name == "test" else train_Re
    with h5py.File(split_dir / f"constants.h5", "w") as file:
        # Same contents as the raw constants.h5 (mesh centers, indices, vertices and attributes), plus the cases
        with h5py.File(dataset_path / "constants.h5", "r") as source:
            for key in source:
                source.copy(source[key], file)
            file.attrs.update(source.attrs)
        cases_group = file.create_group("cases")
        cases_group.create_dataset("Re", data=np.array(Re, dtype=np.float32))

    for i, case in enumerate(split):
        with h5py.File(split_dir / f"{i}.h5", "w") as file:
            file.create_dataset("scalars", data=case["scalars"], compression="gzip", shuffle=True)
            file.create_dataset("fields", data=case["fields"], compression="gzip", shuffle=True)

# Size of the published split folders (train/ + test/ HDF5 files), in decimal gigabytes
metadata["sizeGb"] = round(sum(path.stat().st_size for path in output_dir.rglob("*.h5")) / 1e9, 4)

metadata_path = output_dir / f"{ID}.json"
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")

In [ ]:
### Mesh file for the website's geometry viewer (same format for every benchmark, see mesh_export.py)
from mesh_export import boundary_faces, label_faces, write_mesh

with h5py.File(dataset_path / "constants.h5", "r") as file:
    vertices = file["vertices"][:]
    indices = file["indices"][:]

CYLINDER_CENTER, CYLINDER_RADIUS = np.array([8.0, 10.0]), 1.0
x_min, y_min = vertices.min(axis=0)
x_max, y_max = vertices.max(axis=0)
boundaries = label_faces(vertices, boundary_faces(indices), [
    ("inlet", lambda m: np.isclose(m[:, 0], x_min)),
    ("outlet", lambda m: np.isclose(m[:, 0], x_max)),
    ("topAndBottom", lambda m: np.isclose(m[:, 1], y_min) | np.isclose(m[:, 1], y_max)),
    ("cylinder", lambda m: np.isclose(np.linalg.norm(m - CYLINDER_CENTER, axis=1), CYLINDER_RADIUS, atol=1e-2)),
])
write_mesh(
    Path.cwd().parent / "assets/meshes/cylinder8000.json", vertices, indices, boundaries, nodes="cells",
    extrusion={"depth": 1, "name": "frontAndBack"},
)